# CSE548 Advanced Network Security - Project 1 - Packet Filter Firewall

Created with Grok Build

A packet filter sits on a dual-homed gateway. One side of the gateway faces a single client. The other side faces an upstream network that can reach 8.8.8.8. The filter lets the client do two things, and it drops the rest: ping that one address, and fetch a web page hosted on the gateway itself.


## Project overview

The gateway enforces the policy with iptables. A packet is compared with the rules in a chain from top to bottom, and the first matching rule wins. Three chains in the filter table do the work.

- INPUT handles packets addressed to the gateway.
- FORWARD handles packets the gateway is passing from one network to the other.
- OUTPUT handles packets the gateway creates itself.

All three chains use policy DROP. A packet that matches nothing is discarded rather than forwarded.

The installed policy is meant to do eight things:

1. The client can ping 8.8.8.8 through the gateway.
2. The client can open the web page hosted on the gateway.
3. The client cannot reach any other address.
4. The client cannot ping the gateway.
5. The gateway cannot ping its own loopback address.
6. The gateway cannot ping the client.
7. The gateway cannot ping 8.8.8.8.
8. Packets the client sends on toward 8.8.8.8 are rewritten so they appear to come from the gateway.

The order of the rules is the policy. A DROP that is reached first hides an ACCEPT written below it. `-A` appends a rule at the bottom of a chain. `-I` inserts a rule at the top. Printing the chain after each change, with `iptables -L`, is how that order stays visible. `--line-numbers` names the rule that `-D` should delete when one of them is wrong. When a chain gets tangled, flushing it and starting over is cleaner than trying to patch a rule that sits in the wrong place.


## Network setup

The client has one interface, `enp0s3`, at 10.0.2.4/24. The gateway has two. `enp0s8` is on the client network at 10.0.2.6/24, and `enp0s3` is on the upstream network at 10.0.1.7/24. 8.8.8.8 is reached from the upstream side. The client never talks to that address directly. Its default route hands the packet to 10.0.2.6, and the gateway decides whether to forward it.

VirtualBox is the hypervisor. The client is attached to one NAT network. The gateway is attached to two, which is what makes the machine dual-homed. Both guests are Ubuntu. The settings panels record Ubuntu 23.04, 8 GB of memory, and a 40 GB disk.

:::{figure} images/client-settings.png
The client has a single adapter, on the private NAT network.
:::

:::{figure} images/server-settings.png
The gateway has two adapters: one on the client network and one on the upstream network.
:::

The client's routing table has two default routes. The one that matters for the lab points at 10.0.2.6, the gateway. The other, via 10.0.2.1, belongs to the VirtualBox NAT engine. `route` prints the lab next hop as the name `_gateway`. `route -n` prints the same route as the address 10.0.2.6.

:::{figure} images/client-routes.png
The client's default route for the lab is 10.0.2.6. A second default, 10.0.2.1, is the VirtualBox NAT gateway.
:::

:::{figure} images/client-interfaces.png
The client address is 10.0.2.4/24 on `enp0s3`.
:::

:::{figure} images/server-interfaces.png
On the gateway, `enp0s3` is 10.0.1.7/24 toward the upstream network and `enp0s8` is 10.0.2.6/24 toward the client.
:::


## Software

VirtualBox supplies the two guests. On the gateway, the packet filter is iptables and the page is served by Apache. Nothing else in the path rewrites or inspects the traffic.


## Project description

### Filter rules

:::{figure} images/filter-rules.png
Filter table after the rules were installed. Every chain has policy DROP.
:::

INPUT drops ICMP from the loopback address and drops ICMP from the client at 10.0.2.4. That second drop is why the client cannot ping the gateway. TCP port 80 from the client is accepted, which is the exception for the web page. A later rule accepts ICMP from anywhere. It does not reopen the client's pings, because those packets already matched the DROP higher in the chain.

FORWARD accepts ICMP to 8.8.8.8 and accepts every protocol from the client to 8.8.8.8. It also accepts TCP from anywhere to anywhere. That last rule is wider than the requirement that the client reach only 8.8.8.8 and the page on the gateway. A forwarded TCP connection to some other host would match it.

OUTPUT drops ICMP from the gateway to anywhere, so the gateway cannot ping the client, the loopback address, or 8.8.8.8. It accepts TCP whose source port is 80, which is Apache answering.

### Address translation

:::{figure} images/nat-rules.png
POSTROUTING masquerades traffic to 8.8.8.8, and a second rule masquerades every other forwarded packet.
:::

Masquerade rewrites the source address to the address the gateway uses on the outgoing interface. From the upstream network, a packet that started at 10.0.2.4 looks like it came from 10.0.1.7. The first rule is the specific case the lab asked for. The second rule does the same translation for anything else that is forwarded.

### Web server

Apache serves a one-line page from `/var/www/html/index.html`. The client requests it at 10.0.2.6, which is the gateway's address on the client network, so the packet is delivered locally and hits the INPUT chain rather than FORWARD.

:::{figure} images/web-page.png
The page stored on the gateway.
:::

:::{figure} images/apache-status.png
Apache is running. The fully-qualified-domain-name warning does not stop it from answering.
:::

:::{figure} images/client-curl.png
The client receives the page from 10.0.2.6.
:::

### Port scans

A TCP scan of every port on the gateway, run from the client, finds port 80 open and reports the other 65,534 ports as filtered. Filtered is what a DROP looks like to a scanner: the probe is not answered, and it is not rejected with a reset either.

:::{figure} images/tcp-scan.png
Only TCP port 80 on the gateway answers the client.
:::

A UDP scan of the same host reports every port as open or filtered. A dropped UDP probe and an open UDP port that simply does not reply look the same, so the scan cannot separate them. What it does show is that the gateway is not sending ICMP port-unreachable errors back to the client.

:::{figure} images/udp-scan.png
The UDP scan gets no replies, so every port is reported open or filtered.
:::

### Ping tests

:::{figure} images/ping-8888.png
The client can ping 8.8.8.8. The forward path and the masquerade are both in use.
:::

:::{figure} images/ping-8844.png
A ping from the client to 8.8.4.4 is lost. That address is not the one allowed destination.
:::

:::{figure} images/ping-gateway.png
The client cannot ping the gateway. INPUT already dropped ICMP from 10.0.2.4.
:::

:::{figure} images/ping-localhost.png
The gateway cannot ping 127.0.0.1. The kernel reports `sendmsg: Operation not permitted`, which is the OUTPUT drop, not a missing route.
:::

:::{figure} images/ping-client.png
The gateway cannot ping the client. The same OUTPUT drop refuses the ICMP.
:::

The OUTPUT chain drops ICMP to anywhere, so a ping from the gateway to 8.8.8.8 is covered by the same rule. The report does not include a separate screenshot of that ping.


## Conclusion

The useful habit was to add one rule, print the chain, and only then add the next. A chain is a list that is read from the top, and a mistake high in the list makes every later exception dead. `--line-numbers` and `-D` are enough to pull one bad rule out. Flushing the chain is the faster repair once several of them are in the wrong order.

The ping results are that ordering made visible. Traffic to 8.8.8.8 matches an ACCEPT and comes back. ICMP from the client to the gateway matches an earlier DROP. ICMP created by the gateway matches the OUTPUT policy and is refused before it leaves the machine.
